# PyTorch MLP Pruning Tutorial (Target Matrix: 32x50)

이 노트북에서는 CNN 대신 **MLP(Fully Connected Network)**를 사용합니다.
특히 사용자 요청에 따라 **32x50 크기의 가중치 행렬**을 포함하는 레이어를 설계하여, 해당 행렬이 학습되고 가지치기(Pruning) 되는 과정을 확인합니다.

**모델 구조:**
1. Input (784) -> Hidden 1 (50)
2. **Hidden 1 (50) -> Hidden 2 (32)** : **[이곳에서 32x50 행렬 연산 발생]**
3. Hidden 2 (32) -> Output (10)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.utils.prune as prune
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Random Seed 및 Device 설정
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# 1. 데이터셋 준비 (MNIST)
# MLP는 1차원 벡터 입력을 받으므로 이미지를 Flatten 할 예정입니다.
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

# 2. MLP 모델 정의
class TargetMLP(nn.Module):
    def __init__(self):
        super(TargetMLP, self).__init__()

        # Layer 1: 784 (28x28) -> 50
        self.fc1 = nn.Linear(784, 50)

        # Layer 2: 50 -> 32
        # PyTorch Linear Weight Shape: (Out_features, In_features)
        # 따라서 이 레이어의 weight는 (32, 50) 크기가 됩니다.
        self.fc2 = nn.Linear(50, 32)

        # Layer 3: 32 -> 10 (Output)
        self.fc3 = nn.Linear(32, 10)

    def forward(self, x):
        # Flatten: (Batch, 1, 28, 28) -> (Batch, 784)
        x = x.view(-1, 28 * 28)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x)) # 여기서 (Batch, 50) x (50, 32)^T 연산 수행
        x = self.fc3(x)
        return x

model = TargetMLP().to(device)

# 3. 32x50 행렬 존재 확인
print("--- 모델 레이어별 가중치 크기 확인 ---")
print(f"fc1 weight shape: {model.fc1.weight.shape}")
print(f"fc2 weight shape: {model.fc2.weight.shape}  <-- 요청하신 32x50 행렬")
print(f"fc3 weight shape: {model.fc3.weight.shape}")

--- 모델 레이어별 가중치 크기 확인 ---
fc1 weight shape: torch.Size([50, 784])
fc2 weight shape: torch.Size([32, 50])  <-- 요청하신 32x50 행렬
fc3 weight shape: torch.Size([10, 32])


In [ ]:
def train_model(model, trainloader, epochs=1):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    print(f"\n--- Training for {epochs} Epoch(s) ---")
    for epoch in range(epochs):
        running_loss = 0.0
        for i, data in enumerate(trainloader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"{epoch} epochs completed")
    print("--- Training Completed ---")

def evaluate_accuracy(model, testloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Model Accuracy: {accuracy:.2f}%')

# 학습 실행
evaluate_accuracy(model, testloader)
train_model(model, trainloader, epochs=10)
evaluate_accuracy(model, testloader)

Model Accuracy: 9.20%

--- Training for 10 Epoch(s) ---
0 epochs completed
1 epochs completed
2 epochs completed
3 epochs completed
4 epochs completed
5 epochs completed
6 epochs completed
7 epochs completed
8 epochs completed
9 epochs completed
--- Training Completed ---
Model Accuracy: 96.58%


In [ ]:
import copy

# 1. 프루닝 전, 깊은 복사(Deep Copy)로 원본 보존
original_model = copy.deepcopy(model)

# 2. 기존 model 변수에 대해 프루닝 실행 (덮어쓰기 됨)
prune.l1_unstructured(model.fc2, name='weight', amount=0.5)

# 3. 결과 확인
print("원본 모델은 그대로인가? (Layer 이름 확인)")
# 원본은 weight가 그대로 파라미터로 존재
print(f"Original: {'weight_orig' in dict(original_model.fc2.named_buffers())}")  # False 출력 예상

print("\n프루닝 모델은 변했는가?")
# 프루닝된 모델은 weight_orig가 버퍼로 존재
print(f"Pruned: {'weight_orig' in dict(model.fc2.named_buffers())}")    # True 출력 예상

원본 모델은 그대로인가? (Layer 이름 확인)
Original: False

프루닝 모델은 변했는가?
Pruned: False


In [ ]:
model = copy.deepcopy(original_model)
# 2. 기존 model 변수에 대해 프루닝 실행 (덮어쓰기 됨)
prune.l1_unstructured(model.fc2, name='weight', amount=0.50)

Linear(in_features=50, out_features=32, bias=True)

In [ ]:
# Pruning 영구 적용 (Mask 제거 및 Weight 행렬 자체를 수정)
prune.remove(model.fc2, 'weight')

print("Pruning applied permanently.")

# 최종적으로 32x50 행렬이 희소(Sparse)해졌는지 확인
print(f"Final fc2 weight shape: {model.fc2.weight.shape}")
evaluate_accuracy(original_model, testloader)
evaluate_accuracy(model, testloader)

Pruning applied permanently.
Final fc2 weight shape: torch.Size([32, 50])
Model Accuracy: 96.58%
Model Accuracy: 96.15%


In [ ]:
import numpy as np

# 1. 데이터를 저장할 리스트와 Hook 함수 정의
captured_input = None
captured_output = None

def get_activation_hook(module, input, output):
    global captured_input, captured_output
    # input은 튜플 형태로 들어오므로 첫 번째 요소를 꺼냅니다.
    captured_input = input[0].detach().cpu()
    captured_output = output.detach().cpu()

# 2. fc2 레이어에 Hook 등록
# 모델이 forward pass를 할 때마다 이 함수가 자동으로 실행됩니다.
hook_handle = model.fc2.register_forward_hook(get_activation_hook)

# 3. 테스트 데이터 하나만 모델에 통과시키기 (Inference)
dummy_input, _ = next(iter(testloader))
dummy_input = dummy_input.to(device)

# 배치 사이즈가 1000이므로, 헷갈리지 않게 첫 번째 이미지(1장)만 잘라서 넣겠습니다.
single_image = dummy_input[0].unsqueeze(0)  # Shape: (1, 1, 28, 28)

print(f"--- Inference Start with Input Shape: {single_image.shape} ---")
model.eval()
with torch.no_grad():
    _ = model(single_image)
print("--- Inference Complete ---\n")

# Hook 제거 (계속 메모리를 차지하지 않도록)
hook_handle.remove()

# 4. 캡처된 데이터 및 가중치 확인
# (1) 가중치 행렬 (Weight Matrix)
weight_matrix = model.fc2.weight.detach().cpu()
# (2) 입력 벡터 (Input Vector)
input_vector = captured_input

print("=== [1] 32x50 가중치 행렬 (Weight) ===")
print(f"Shape: {weight_matrix.shape}") # (32, 50)
print("일부 데이터 확인 (첫 5x5):\n", weight_matrix[:5, :5].numpy())

print("\n=== [2] 레이어로 들어온 입력값 (Input) ===")
print(f"Shape: {input_vector.shape}")   # (1, 50) -> 배치크기 1, 입력특징 50
print("전체 데이터 확인:\n", input_vector.numpy())

print("\n=== [3] 연산 결과 (Output = Input x Weight^T + Bias) ===")
print(f"실제 출력 Shape: {captured_output.shape}") # (1, 32)
print("일부 데이터 확인 (첫 5개):\n", captured_output[0, :5].numpy())

--- Inference Start with Input Shape: torch.Size([1, 1, 28, 28]) ---
--- Inference Complete ---

=== [1] 32x50 가중치 행렬 (Weight) ===
Shape: torch.Size([32, 50])
일부 데이터 확인 (첫 5x5):
 [[-0.138353   -0.         -0.         -0.12024175  0.        ]
 [-0.11409431 -0.          0.          0.17923476  0.25899857]
 [ 0.          0.         -0.39237878  0.18546991 -0.        ]
 [-0.          0.0986955   0.         -0.14496613  0.11546534]
 [ 0.          0.11195028  0.         -0.15840027  0.09526892]]

=== [2] 레이어로 들어온 입력값 (Input) ===
Shape: torch.Size([1, 50])
전체 데이터 확인:
 [[ 0.          0.          3.211437    0.1765825   2.6574323   9.532088
   0.          0.          0.          0.          9.909831    0.
  15.64244     0.          0.         14.6862135   8.450303    0.
   0.          0.          0.          0.8532652   0.         10.895899
   0.          0.          0.          2.626284    0.          9.472856
   0.          0.          0.          0.          5.526463    0.
   0.9061959   0. 

In [ ]:
import torch
import torch.nn.functional as F

# 1. 양자화 시뮬레이션 함수 (Fake Quantization)
def quantize_simulate(tensor, frac_bits=4):
    """
    텐서의 값을 fx8(Qn.m) 포맷 범위로 강제 변환합니다.
    (실수형 데이터지만, 값은 8비트 해상도로 끊어짐)
    """
    total_bits = 8
    scale = 2 ** frac_bits
    min_val = -(2 ** (total_bits - 1))
    max_val = (2 ** (total_bits - 1)) - 1

    # Scale -> Round -> Clamp
    x = tensor * scale
    x = torch.round(x)
    x = torch.clamp(x, min_val, max_val)

    # De-scale (원래 크기로 복구하지만, 정밀도는 떨어져 있음)
    x = x / scale
    return x

# 2. Fx8 전용 추론(Inference) 함수
def eval_fx8_accuracy(model, testloader, frac_bits=4):
    model.eval()
    correct = 0
    total = 0

    print(f"--- Fx8 (Q3.{frac_bits}) Accuracy Check Start ---")

    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            # [전처리] 입력 이미지 Flatten
            x = images.view(-1, 28 * 28)

            # === Layer 1 Simulation ===
            # 1. 입력값(Activation) 양자화
            x_q = quantize_simulate(x, frac_bits)
            # 2. 가중치(Weight) 양자화
            w1_q = quantize_simulate(model.fc1.weight, frac_bits)
            # 3. 연산 (MatMul + Bias + ReLU)
            # 주의: Bias는 보통 하드웨어에서 32bit accumulator를 쓰므로 양자화 안 하거나 덜 함.
            # 여기서는 편의상 Bias는 고정밀도 유지.
            out1 = F.linear(x_q, w1_q, model.fc1.bias)
            out1 = F.relu(out1)

            # === Layer 2 Simulation (32x50 Matrix) ===
            # 이전 레이어의 출력(out1)이 다음 레이어의 입력이 되기 전에 양자화됨 (Wire가 8bit이므로)
            out1_q = quantize_simulate(out1, frac_bits)
            w2_q = quantize_simulate(model.fc2.weight, frac_bits)

            out2 = F.linear(out1_q, w2_q, model.fc2.bias)
            out2 = F.relu(out2)

            # === Layer 3 Simulation (Output) ===
            out2_q = quantize_simulate(out2, frac_bits)
            w3_q = quantize_simulate(model.fc3.weight, frac_bits)

            out3 = F.linear(out2_q, w3_q, model.fc3.bias)
            # 출력단은 보통 Softmax/Argmax만 하므로 양자화 굳이 안 해도 됨 (Logits)

            # [평가]
            _, predicted = torch.max(out3.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Fx8 Simulated Accuracy: {accuracy:.2f}%")
    return accuracy

# 3. 비교 실행
# (1) 일반 Float 모델 정확도
print("Original Float Accuracy:")
evaluate_accuracy(original_model, testloader) # 기존 함수 사용

print("-" * 30)

# (2) Fx8 변환 시뮬레이션 정확도 (Q3.4 포맷 가정)
fx8_acc = eval_fx8_accuracy(original_model, testloader, frac_bits=4)
fx8_acc = eval_fx8_accuracy(model, testloader, frac_bits=4)

Original Float Accuracy:
Model Accuracy: 96.58%
------------------------------
--- Fx8 (Q3.4) Accuracy Check Start ---
Fx8 Simulated Accuracy: 93.62%
--- Fx8 (Q3.4) Accuracy Check Start ---
Fx8 Simulated Accuracy: 92.35%


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# 설정 (앞선 정확도 테스트와 동일하게 유지)
FRAC_BITS = 4
SCALE = 2 ** FRAC_BITS

def to_fx8_int_hex(tensor, frac_bits=4):
    """
    텐서를 입력받아 Fx8 범위의 정수(Integer)와 Hex 문자열 리스트를 반환합니다.
    """
    total_bits = 8
    scale = 2 ** frac_bits
    min_val = -(2 ** (total_bits - 1))     # -128
    max_val = (2 ** (total_bits - 1)) - 1  # 127

    # 1. Scaling & Rounding (실수 -> 정수 변환)
    scaled = tensor * scale
    rounded = torch.round(scaled).long()

    # 2. Clipping (8비트 범위 초과 방지)
    clamped = torch.clamp(rounded, min_val, max_val)

    # 3. Hex 변환 (2의 보수 표현)
    hex_list = []
    for val in clamped.view(-1).tolist():
        hex_val = f"{val & 0xFF:02x}" # 8비트 마스킹 후 16진수 변환
        hex_list.append(hex_val)

    return clamped, hex_list

def extract_fx8_data(model, testloader, frac_bits=4):
    model.eval()

    # 테스트셋에서 이미지 1장만 가져오기
    data_iter = iter(testloader)
    images, labels = next(data_iter)

    # (1, 784) 형태로 준비
    x = images[0].view(-1, 28 * 28).to(device)

    print(f"--- Extracting Fx8 Data (Q3.{frac_bits}) ---")

    with torch.no_grad():
        # === [Step 1] Layer 1 직접 연산 (시뮬레이션) ===
        # 입력 이미지 양자화
        x_int, _ = to_fx8_int_hex(x, frac_bits)
        x_sim = x_int.float() / (2**frac_bits) # 다시 float 스케일로 돌려서 연산 (Fake Quant)

        # FC1 가중치 양자화
        w1_int, _ = to_fx8_int_hex(model.fc1.weight, frac_bits)
        w1_sim = w1_int.float() / (2**frac_bits)

        # FC1 연산 & ReLU
        out1 = F.linear(x_sim, w1_sim, model.fc1.bias)
        out1 = F.relu(out1)

        # === [Step 2] Layer 2 (Target: 32x50) 입력값 준비 ===
        # FC1의 출력이 FC2로 들어가기 전에 다시 양자화됩니다. (Activation Quantization)
        # 이것이 바로 Verilog 모듈의 '입력 포트'로 들어갈 값입니다.
        fc2_input_float = out1
        fc2_input_int, fc2_input_hex = to_fx8_int_hex(fc2_input_float, frac_bits)

        # === [Step 3] Layer 2 가중치 준비 ===
        # 이것이 Verilog 모듈의 'ROM/RAM'에 저장될 값입니다.
        fc2_weight_float = model.fc2.weight
        fc2_weight_int, fc2_weight_hex = to_fx8_int_hex(fc2_weight_float, frac_bits)

        # === [Step 4] 검증용 Golden Output 계산 (Integer 레벨) ===
        # 하드웨어 동작과 똑같이 정수끼리 곱해서 계산
        # Input(Integer) * Weight(Integer) = Accumulator(Integer)
        # (1, 50) @ (50, 32) = (1, 32)
        golden_output_int = torch.matmul(fc2_input_int.float(), fc2_weight_int.float().t())

        # Bias 더하기 (Bias도 양자화해서 더해야 완벽하지만, 여기서는 스케일 맞춰서 더함)
        # Bias Scale: 보통 Weight Scale * Input Scale = 2^4 * 2^4 = 2^8
        bias_scale = (2**frac_bits) * (2**frac_bits)
        bias_int = torch.round(model.fc2.bias * bias_scale)
        golden_output_int += bias_int

    return fc2_input_hex, fc2_weight_hex, golden_output_int

# 1. 추출 실행
input_hex, weight_hex, golden_out = extract_fx8_data(model, testloader, FRAC_BITS)

# 2. 파일 저장 (Verilog용)
with open("fc2_input_fx8.mem", "w") as f:
    f.write("\n".join(input_hex))

with open("fc2_weight_fx8.mem", "w") as f:
    f.write("\n".join(weight_hex))

# 3. 결과 확인
print(f"\n[Saved] fc2_input_fx8.mem (Lines: {len(input_hex)})")
print(f"[Saved] fc2_weight_fx8.mem (Lines: {len(weight_hex)})")

print("\n=== Data Check ===")
print(f"Input Hex Example (First 5): {input_hex[:5]}")
print(f"Weight Hex Example (First 5): {weight_hex[:5]}")
print(f"Golden Output (Integer) Example (First 5): {golden_out[0][:5].long().tolist()}")

--- Extracting Fx8 Data (Q3.4) ---

[Saved] fc2_input_fx8.mem (Lines: 50)
[Saved] fc2_weight_fx8.mem (Lines: 1600)

=== Data Check ===
Input Hex Example (First 5): ['00', '00', '2f', '00', '2c']
Weight Hex Example (First 5): ['fe', '00', '00', 'fe', '00']
Golden Output (Integer) Example (First 5): [1410, 1380, 82, 1083, 1424]


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# 설정
FRAC_BITS = 4
SCALE = 2 ** FRAC_BITS

def to_fx8_int_hex(tensor, frac_bits=4):
    """
    [입력/가중치용] 8비트(Fx8) 정수 및 Hex 변환
    """
    total_bits = 8
    scale = 2 ** frac_bits
    min_val = -(2 ** (total_bits - 1))     # -128
    max_val = (2 ** (total_bits - 1)) - 1  # 127

    scaled = tensor * scale
    rounded = torch.round(scaled).long()
    clamped = torch.clamp(rounded, min_val, max_val)

    hex_list = []
    for val in clamped.view(-1).tolist():
        hex_list.append(f"{val & 0xFF:02x}") # 8비트 Hex

    return clamped, hex_list

def to_fx32_hex(tensor):
    """
    [결과값용] 32비트(Accumulator) Hex 변환 (2의 보수 처리)
    """
    hex_list = []
    for val in tensor.view(-1).tolist():
        # 32비트 마스킹 (음수 처리를 위해 필수)
        hex_list.append(f"{int(val) & 0xFFFFFFFF:08x}")
    return hex_list

def extract_fx8_data(model, testloader, frac_bits=4):
    model.eval()

    # 테스트셋에서 이미지 1장만 가져오기
    data_iter = iter(testloader)
    images, labels = next(data_iter)

    x = images[0].view(-1, 28 * 28).to(device)

    print(f"--- Extracting Fx8 Data & Golden Output (Hex) ---")

    with torch.no_grad():
        # === [Step 1] Layer 1 시뮬레이션 ===
        x_int, _ = to_fx8_int_hex(x, frac_bits)
        x_sim = x_int.float() / (2**frac_bits)

        w1_int, _ = to_fx8_int_hex(model.fc1.weight, frac_bits)
        w1_sim = w1_int.float() / (2**frac_bits)

        out1 = F.relu(F.linear(x_sim, w1_sim, model.fc1.bias))

        # === [Step 2] Layer 2 입력값 (Input) ===
        fc2_input_float = out1
        fc2_input_int, fc2_input_hex = to_fx8_int_hex(fc2_input_float, frac_bits)

        # === [Step 3] Layer 2 가중치 (Weight) ===
        fc2_weight_float = model.fc2.weight
        fc2_weight_int, fc2_weight_hex = to_fx8_int_hex(fc2_weight_float, frac_bits)

        # === [Step 4] Golden Output 계산 (Integer) ===
        # (1, 50) x (50, 32) = (1, 32)
        golden_output_int = torch.matmul(fc2_input_int.float(), fc2_weight_int.float().t())

        # Bias 더하기 (Scale = 2^4 * 2^4 = 2^8)
        bias_scale = (2**frac_bits) * (2**frac_bits)
        bias_int = torch.round(model.fc2.bias * bias_scale)

        golden_output_int += bias_int
        golden_output_int = golden_output_int.long() # 정수형 변환

        # === [Step 5] Golden Output을 Hex로 변환 (NEW) ===
        golden_output_hex = to_fx32_hex(golden_output_int)

    return fc2_input_hex, fc2_weight_hex, golden_output_hex

# 1. 추출 실행
input_hex, weight_hex, golden_hex = extract_fx8_data(model, testloader, FRAC_BITS)

# # 2. 파일 저장 (Verilog $readmemh용)
# with open("fc2_input_fx8.mem", "w") as f:
#     f.write("\n".join(input_hex))
# with open("fc2_input_fx8.txt", "w") as f:
#     f.write("\n".join(input_hex))


# with open("fc2_weight_fx8.mem", "w") as f:
#     f.write("\n".join(weight_hex))
# with open("fc2_weight_fx8.txt", "w") as f:
#     f.write("\n".join(weight_hex))

# with open("fc2_golden_fx8.mem", "w") as f:
#     f.write("\n".join(golden_hex))
# with open("fc2_golden_fx8.txt", "w") as f:
#     f.write("\n".join(golden_hex))


# 3. 결과 확인
print(f"\n[Saved] fc2_input_fx8.mem  (Lines: {len(input_hex)})")
print(f"[Saved] fc2_weight_fx8.mem (Lines: {len(weight_hex)})")
print(f"[Saved] fc2_golden_fx8.mem (Lines: {len(golden_hex)}) -> Golden Output Hex File")

print("\n=== Data Check ===")
print(f"Input Hex (First 5):  {input_hex[:5]}")
print(f"Weight Hex (First 5): {weight_hex[:5]}")
print(f"Golden Hex (First 5): {golden_hex[:5]}")

--- Extracting Fx8 Data & Golden Output (Hex) ---

[Saved] fc2_input_fx8.mem  (Lines: 50)
[Saved] fc2_weight_fx8.mem (Lines: 1600)
[Saved] fc2_golden_fx8.mem (Lines: 32) -> Golden Output Hex File

=== Data Check ===
Input Hex (First 5):  ['00', '00', '2f', '00', '2c']
Weight Hex (First 5): ['fe', '00', '00', 'fe', '00']
Golden Hex (First 5): ['00000582', '00000564', '00000052', '0000043b', '00000590']


In [ ]:
import torch
import numpy as np
import os

# === [1] 사용자 제공 클래스 (수정 없이 사용하거나 약간 보완) ===
class SpMMDataProcessor:
    """
    행렬을 CSR 포맷 변환 및 Dense 패킹을 수행하는 클래스
    """
    def __init__(self, rows=32, cols=50):
        self.rows = rows
        self.cols = cols

    def to_csr(self, sparse_matrix):
        """
        Sparse Matrix (Weights) -> CSR Format
        Output:
          - row_ptr: 행의 시작 지점 인덱스
          - csr_data: (Value << 8 | Col_Idx) 형태로 패킹된 16비트 데이터 리스트
        """
        row_ptr = [0]
        csr_data = []

        # sparse_matrix는 numpy array (int8) 가정
        for row in range(self.rows):
            nnz_in_row = 0
            for col in range(self.cols):
                val = sparse_matrix[row, col]
                if val != 0:
                    # 16비트 패킹: [Value(8bit) | ColIdx(8bit)]
                    # & 0xFF 처리를 통해 음수도 올바르게 2의 보수 비트로 변환
                    packed_val = ((int(val) & 0xFF) << 8) | (col & 0xFF)
                    csr_data.append(packed_val)
                    nnz_in_row += 1
            row_ptr.append(row_ptr[-1] + nnz_in_row)

        return row_ptr, csr_data

    def pack_dense_vector(self, dense_vector):
        """
        Dense Vector (Inputs) -> Packed 16-bit
        32개의 8-bit 원소를 16비트 워드 16개로 패킹 (여기서는 50개 -> 25개)
        Logic: [vec[j+1] | vec[j]]
        """
        packed_dense = []
        # 길이가 홀수일 경우를 대비해 짝수로 맞춤 (Padding 0)
        if len(dense_vector) % 2 != 0:
            dense_vector = np.append(dense_vector, 0)

        for j in range(0, len(dense_vector), 2):
            low_byte = int(dense_vector[j]) & 0xFF
            high_byte = int(dense_vector[j+1]) & 0xFF
            packed_word = (high_byte << 8) | low_byte
            packed_dense.append(packed_word)
        return packed_dense

# === [2] PyTorch 데이터 추출 및 변환 실행 함수 ===
def process_and_save_data(model, testloader, frac_bits=4):
    device = next(model.parameters()).device
    model.eval()

    # 1. 데이터 추출 (PyTorch -> Int Tensor)
    images, _ = next(iter(testloader))
    x = images[0].view(-1, 28 * 28).to(device)

    with torch.no_grad():
        # Layer 1 시뮬레이션 (Input 값을 만들기 위함)
        scale = 2 ** frac_bits
        x_scaled = torch.round(x * scale)
        x_clamped = torch.clamp(x_scaled, -128, 127)
        x_sim = x_clamped.float() / scale

        w1_scaled = torch.round(model.fc1.weight * scale)
        w1_clamped = torch.clamp(w1_scaled, -128, 127)
        w1_sim = w1_clamped.float() / scale

        out1 = torch.nn.functional.relu(torch.nn.functional.linear(x_sim, w1_sim, model.fc1.bias))

        # --- Target Data 준비 (Layer 2) ---
        # (1) 입력 벡터 (Input Vector) -> Quantized Int8
        input_float = out1.view(-1) # 1D Array
        input_int = torch.clamp(torch.round(input_float * scale), -128, 127).long().cpu().numpy()

        # (2) 가중치 행렬 (Weight Matrix) -> Quantized Int8
        weight_float = model.fc2.weight
        weight_int = torch.clamp(torch.round(weight_float * scale), -128, 127).long().cpu().numpy()

    # 2. SpMMDataProcessor 사용
    rows, cols = weight_int.shape # (32, 50)
    processor = SpMMDataProcessor(rows, cols)

    # A. Weight CSR 변환
    print(f"--- Converting Weight (32x50) to CSR ---")
    # 주의: Pruning이 안 된 모델이라면 0이 거의 없어서 모든 데이터가 변환될 수 있음
    row_ptr, csr_data = processor.to_csr(weight_int)

    # B. Input Vector Packing
    print(f"--- Packing Input Vector (Length: {len(input_int)}) ---")
    packed_input = processor.pack_dense_vector(input_int)

    # # 3. 파일 저장 (Hex 포맷)

    # # (1) CSR Row Pointer (인덱스 값이므로 32bit로 넉넉하게 저장하거나 필요시 조절)
    # with open("csr_row_ptr.mem", "w") as f:
    #     for val in row_ptr:
    #         f.write(f"{val:08x}\n") # Verilog 가독성을 위해 8자리 Hex
    # with open("csr_row_ptr.txt", "w") as f:
    #     for val in row_ptr:
    #         f.write(f"{val:08x}\n") # Verilog 가독성을 위해 8자리 Hex

    # # (2) CSR Data (Value | Col) -> 16bit Hex
    # with open("csr_data.mem", "w") as f:
    #     for val in csr_data:
    #         f.write(f"{val:04x}\n")
    # with open("csr_data.txt", "w") as f:
    #     for val in csr_data:
    #         f.write(f"{val:04x}\n")

    # # (3) Packed Input (Val H | Val L) -> 16bit Hex
    # with open("packed_input.mem", "w") as f:
    #     for val in packed_input:
    #         f.write(f"{val:04x}\n")

    # print("\n=== File Generation Complete ===")
    # print(f"1. csr_row_ptr.mem: {len(row_ptr)} lines (Start indices)")
    # print(f"2. csr_data.mem:    {len(csr_data)} lines (Non-zero elements)")
    # print(f"3. packed_input.mem:{len(packed_input)} lines (Packed 2 inputs per line)")

    # 검증용 출력
    print("\n=== Sample Check ===")
    print(f"Row Ptrs (First 5): {row_ptr[:5]}")
    if len(csr_data) > 0:
        val = csr_data[0]
        print(f"CSR Data[0] (Hex): {val:04x} -> Value: {np.int8(val >> 8)}, Col: {val & 0xFF}")
    if len(packed_input) > 0:
        val = packed_input[0]
        print(f"Packed Input[0] (Hex): {val:04x} -> High: {np.int8(val >> 8)}, Low: {val & 0xFF}")
    return row_ptr, csr_data, packed_input, weight_int

# 실행
row_ptr, csr_data, packed_input, weight_int, = process_and_save_data(model, testloader, FRAC_BITS)

--- Converting Weight (32x50) to CSR ---
--- Packing Input Vector (Length: 50) ---

=== Sample Check ===
Row Ptrs (First 5): [0, 27, 60, 87, 115]


OverflowError: Python integer 254 out of bounds for int8

In [ ]:
import os
import torch
import numpy as np

# 저장 경로 설정 (현재 폴더)
PACKET_DIR = "."
if not os.path.exists(PACKET_DIR):
    os.makedirs(PACKET_DIR)

# === 1. UART Packet Generator Class (사용자 제공 코드) ===
class UARTPacketGenerator:
    """
    UART 패킷 생성기
    Protocol: [Header(8bit)] [CMD(4bit)+ID(4bit)] [Len_L(8bit)] [Len_H(8bit)] [Data(16bit)...]
    """
    def __init__(self, header=0xAA):
        self.header = header
        self.CMD_WRITE = 0x1
        self.CMD_READ  = 0x2
        self.CMD_RESET = 0x3
        self.CMD_START = 0x4
        self.CMD_ECHO  = 0x5
        self.CMD_PERF  = 0x6

    def _pack_cmd_id(self, cmd, target_id):
        return ((cmd & 0x0F) << 4) | (target_id & 0x0F)

    def make_packet(self, cmd, target_id, length_val=0, data_list=None):
        """
        패킷 바이트 리스트 생성 함수
        """
        packet = []
        packet.append(self.header)
        packet.append(self._pack_cmd_id(cmd, target_id))

        # 1. Length 결정 (데이터가 있으면 데이터 길이, 없으면 입력된 길이값 사용)
        actual_len = len(data_list) if data_list else length_val

        # Length는 16비트로 쪼개서 저장 (Little Endian: Low -> High)
        packet.append(actual_len & 0xFF)
        packet.append((actual_len >> 8) & 0xFF)

        # 2. Payload 추가 (Little Endian)
        if data_list:
            for data in data_list:
                packet.append(data & 0xFF)        # LSB
                packet.append((data >> 8) & 0xFF) # MSB
        return packet

    def save_hex_file(self, filename, byte_stream):
        """리스트를 txt 파일로 저장 (주석 없이 데이터만)"""
        path = os.path.join(PACKET_DIR, filename)
        with open(path, 'w') as f:
            for byte in byte_stream:
                f.write(f"{byte:02X}\n")
        print(f"💾 {filename:<25} 저장 완료 ({len(byte_stream)} bytes)")


# === 2. 이전 단계 데이터 준비 (CSR & Packing) ===
# (문맥상 이전에 생성된 리스트 변수들이 있다고 가정합니다.
# 만약 없다면, 앞선 코드의 실행 결과인 리스트를 여기에 직접 할당해야 합니다.)
# 여기서는 예시를 위해 앞선 코드의 변수명을 그대로 사용합니다.

# 예시 데이터 (앞선 코드 실행 후 메모리에 있는 값들)
# row_ptr: 리스트 [0, 5, 10, ...]
# csr_data: 리스트 [0x1234, 0x5678, ...] (16bit packed)
# packed_input: 리스트 [0xabcd, ...] (16bit packed)

# 만약 코드를 처음부터 돌린다면 아래 변수들이 process_and_save_data 함수에서 반환되어야 합니다.
# 여기서는 이해를 돕기 위해 앞선 함수의 결과를 받는 래퍼 함수를 만듭니다.

def generate_and_save_uart_packets(row_ptr, csr_data, packed_input):
    uart_gen = UARTPacketGenerator()

    print("\n=== UART Packet Generation Start ===")

    # [Packet 1] CSR Row Pointer
    # Target ID: 0x1 (가정: 하드웨어의 Row Ptr 메모리 ID)
    # CMD: WRITE (0x1)
    pkt_row_ptr = uart_gen.make_packet(
        cmd=uart_gen.CMD_WRITE,
        target_id=0x1,
        data_list=row_ptr
    )
    uart_gen.save_hex_file("uart_pkt_row_ptr.hex", pkt_row_ptr)
    uart_gen.save_hex_file("uart_pkt_row_ptr.txt", pkt_row_ptr)

    # [Packet 2] CSR Data (Value | Col)
    # Target ID: 0x2 (가정: 하드웨어의 CSR Data 메모리 ID)
    pkt_csr_data = uart_gen.make_packet(
        cmd=uart_gen.CMD_WRITE,
        target_id=0x2,
        data_list=csr_data
    )
    uart_gen.save_hex_file("uart_pkt_csr_data.hex", pkt_csr_data)
    uart_gen.save_hex_file("uart_pkt_csr_data.txt", pkt_csr_data)

    # [Packet 3] Input Vector (Packed)
    # Target ID: 0x3 (가정: 하드웨어의 Input Vector 메모리 ID)
    pkt_input = uart_gen.make_packet(
        cmd=uart_gen.CMD_WRITE,
        target_id=0x3,
        data_list=packed_input
    )
    uart_gen.save_hex_file("uart_pkt_input.hex", pkt_input)
    uart_gen.save_hex_file("uart_pkt_input.txt", pkt_input)

    # [Optional] Start Command Packet
    # 데이터 전송 후 연산 시작 명령
    # Target ID: 0x0 (System Control)
    pkt_start = uart_gen.make_packet(
        cmd=uart_gen.CMD_START,
        target_id=0x0,
        length_val=0, # Payload 없음
        data_list=None
    )
    uart_gen.save_hex_file("uart_pkt_start.hex", pkt_start)
    uart_gen.save_hex_file("uart_pkt_start.txt", pkt_start)

    print("=== All Packets Generated ===")



In [ ]:
# === 실행부 ===
# 앞선 코드의 변수(row_ptr, csr_data, packed_input)가 필요합니다.
# 편의상 앞의 SpMMDataProcessor 로직을 통해 데이터를 다시 확보하고 실행합니다.

# (앞선 코드의 process_and_save_data 함수가 리스트 자체를 반환하도록 살짝 수정했다고 가정하거나,
#  혹은 저장된 .mem 파일을 읽어서 리스트로 만드는 방법이 있습니다.
#  가장 깔끔한 건 앞선 코드 실행 시 변수에 담아두는 것입니다.)

# 예시: 앞선 코드의 변수들을 그대로 전달
# 실제 실행 시에는 아래 row_ptr, csr_data, packed_input 에
# SpMMDataProcessor로 만든 정수 리스트가 들어가야 합니다.

# [Test를 위한 Dummy Data 생성 - 실제 사용 시에는 이 부분을 지우고 위에서 만든 데이터를 넣으세요]
# if 'row_ptr' not in locals():
#     print("Warning: 앞선 데이터가 없어 임시 데이터를 사용합니다.")
#     row_ptr = [0, 2, 5]
#     csr_data = [0x1234, 0x5678, 0x9abc, 0xdef0, 0x1122]
#     packed_input = [0x0001, 0x0203]

# 함수 호출
if 'row_ptr' in locals() and 'csr_data' in locals() and 'packed_input' in locals():
    # 앞선 코드에서 생성된 실제 데이터 사용
    # 주의: row_ptr 등이 numpy 타입이면 tolist()로 파이썬 리스트 변환이 필요할 수 있습니다.

    # 타입 안전 변환
    def to_list(data):
        if isinstance(data, np.ndarray) or isinstance(data, torch.Tensor):
            return data.tolist()
        return data

    generate_and_save_uart_packets(
        to_list(row_ptr),
        to_list(csr_data),
        to_list(packed_input) # 여기서는 packed_input 변수명 사용 (SpMMDataProcessor 결과)
    )
else:
    print("Error: 변수 row_ptr, csr_data, packed_input이 정의되지 않았습니다. 앞선 셀을 먼저 실행해주세요.")

In [ ]:
import torch
import numpy as np
import os
import random

# 저장 경로 설정
PACKET_DIR = "uart_packets_generated"
if not os.path.exists(PACKET_DIR):
    os.makedirs(PACKET_DIR)

# ====================================================
# 1. Helper Classes & Functions
# ====================================================

class UARTPacketGenerator:
    """ UART 패킷 생성기 (사용자 정의 프로토콜 준수) """
    def __init__(self, header=0xAA):
        self.header = header
        self.CMD_WRITE = 0x1
        self.CMD_READ  = 0x2
        self.CMD_RESET = 0x3
        self.CMD_START = 0x4
        self.CMD_ECHO  = 0x5
        self.CMD_PERF  = 0x6

    def _pack_cmd_id(self, cmd, target_id):
        return ((cmd & 0x0F) << 4) | (target_id & 0x0F)

    def make_packet(self, cmd, target_id, length_val=0, data_list=None):
        packet = []
        packet.append(self.header)
        packet.append(self._pack_cmd_id(cmd, target_id))

        actual_len = len(data_list) if data_list else length_val
        packet.append(actual_len & 0xFF)
        packet.append((actual_len >> 8) & 0xFF)

        if data_list:
            for data in data_list:
                packet.append(data & 0xFF)
                packet.append((data >> 8) & 0xFF)
        return packet

    def save_hex_file(self, filename, byte_stream):
        path = os.path.join(PACKET_DIR, filename)
        with open(path, 'w') as f:
            for byte in byte_stream:
                f.write(f"{byte:02X}\n")
        print(f"💾 {filename:<25} 저장 완료 ({len(byte_stream)} bytes)")

class SpMMDataProcessor:
    """ 데이터 변환 및 패킹 클래스 """
    def __init__(self, rows=32, cols=50):
        self.rows = rows
        self.cols = cols

    def to_csr(self, sparse_matrix):
        row_ptr = [0]
        csr_data = []
        for row in range(self.rows):
            nnz_in_row = 0
            for col in range(self.cols):
                val = sparse_matrix[row, col]
                if val != 0:
                    packed_val = ((int(val) & 0xFF) << 8) | (col & 0xFF)
                    csr_data.append(packed_val)
                    nnz_in_row += 1
            row_ptr.append(row_ptr[-1] + nnz_in_row)
        return row_ptr, csr_data

    def pack_dense_vector(self, dense_vector):
        """ 1D 벡터 패킹 (2 elements -> 1 word) """
        packed = []
        vec = dense_vector.tolist() if isinstance(dense_vector, np.ndarray) else dense_vector
        if len(vec) % 2 != 0: vec.append(0) # Padding

        for j in range(0, len(vec), 2):
            low = int(vec[j]) & 0xFF
            high = int(vec[j+1]) & 0xFF
            packed.append((high << 8) | low)
        return packed

    def pack_dense_matrix(self, matrix):
        """ 2D 매트릭스 패킹 (Dense Weight Test용) """
        packed = []
        # Row-Major 순회하며 2개씩 묶음
        flat = matrix.flatten()
        return self.pack_dense_vector(flat)

def convert_byte_to_word_format(input_file_, bytes_per_word=2):
    """ 바이트 파일을 읽어 사람이 보기 편한 메모리 뷰 생성 """
    input_path = os.path.join(PACKET_DIR, input_file_)
    if not os.path.exists(input_path):
        print(f"❌ 파일 없음: {input_path}")
        return

    output_file = input_path.replace(".txt", "_MEM_VIEW.txt")

    try:
        with open(input_path, 'r') as f_in, open(output_file, 'w') as f_out:
            lines = [line.strip() for line in f_in.readlines() if line.strip()]

            f_out.write(f"# Memory View for {input_file_}\n")
            f_out.write(f"# Width: {bytes_per_word * 8}-bit\n")
            f_out.write(f"# {'Address':<8} | {'Hex Value':<10} | {'Decimal'}\n")
            f_out.write("-" * 45 + "\n")

            word_count = 0
            for i in range(0, len(lines), bytes_per_word):
                chunks = lines[i : i + bytes_per_word]
                if len(chunks) < bytes_per_word: break

                # Little Endian Logic
                hex_str = ""
                for b in reversed(chunks): hex_str += b

                # Signed Int Check (32bit vs 16bit)
                val_int = int(hex_str, 16)
                if bytes_per_word == 4 and val_int > 0x7FFFFFFF:
                    val_int -= 0x100000000
                elif bytes_per_word == 2 and val_int > 0x7FFF:
                    val_int -= 0x10000

                f_out.write(f"@{word_count:04X}     | {hex_str}       | {val_int}\n")
                word_count += 1

        print(f"✅ 뷰 생성 완료: {os.path.basename(output_file)}")
    except Exception as e:
        print(f"❌ 뷰 생성 실패: {e}")

# ====================================================
# 2. Main Logic: PyTorch Data -> Packets
# ====================================================

def generate_full_test_scenario(model, testloader, frac_bits=4):
    gen = UARTPacketGenerator()
    device = next(model.parameters()).device
    model.eval()

    # --- [Step 1] PyTorch 데이터 추출 ---
    print("\n[Step 1] PyTorch 모델에서 데이터 추출 및 양자화...")
    images, _ = next(iter(testloader))
    x = images[0].view(-1, 28 * 28).to(device)
    scale = 2 ** frac_bits

    with torch.no_grad():
        # Layer 1 Sim
        x_int = torch.clamp(torch.round(x * scale), -128, 127)
        x_sim = x_int.float() / scale
        w1_int = torch.clamp(torch.round(model.fc1.weight * scale), -128, 127)
        w1_sim = w1_int.float() / scale
        out1 = torch.nn.functional.relu(torch.nn.functional.linear(x_sim, w1_sim, model.fc1.bias))

        # Layer 2 Input (Vector B)
        input_vec_int = torch.clamp(torch.round(out1.view(-1) * scale), -128, 127).long().cpu().numpy()

        # Layer 2 Weight (Matrix A)
        weight_mat_int = torch.clamp(torch.round(model.fc2.weight * scale), -128, 127).long().cpu().numpy()

        # Golden Result (Ref C)
        # (1, 50) x (50, 32) + Bias -> 32 elements
        # Integer MAC Operation
        golden_res_int = np.dot(input_vec_int, weight_mat_int.T)
        bias_int = torch.round(model.fc2.bias * (scale**2)).detach().cpu().numpy()
        ref_c = (golden_res_int + bias_int).astype(np.int32)

    # --- [Step 2] 데이터 패킹 (CSR, Dense) ---
    print("[Step 2] 데이터를 하드웨어 포맷으로 변환...")
    rows, cols = weight_mat_int.shape
    proc = SpMMDataProcessor(rows, cols)

    # 1. CSR Data
    row_ptr, csr_data = proc.to_csr(weight_mat_int)

    # 2. Dense Packed Weight (Optional: 'd' suffix)
    packed_sparse_a = proc.pack_dense_matrix(weight_mat_int)

    # 3. Packed Input Vector
    packed_dense = proc.pack_dense_vector(input_vec_int)

    # --- Helper: Expect 파일 저장 ---
    def save_expect_file(filename, data_list, is_32bit=False):
        byte_stream = []
        for val in data_list:
            val = int(val)
            byte_stream.append(val & 0xFF)
            byte_stream.append((val >> 8) & 0xFF)
            if is_32bit:
                byte_stream.append((val >> 16) & 0xFF)
                byte_stream.append((val >> 24) & 0xFF)
        gen.save_hex_file(filename, byte_stream)
        # .mem 파일도 동일하게 생성
        gen.save_hex_file(filename.replace(".txt", ".mem"), byte_stream)


    # ====================================================
    # [Step 3] UART 패킷 생성 시나리오
    # ====================================================
    print("\n[Step 3] UART 패킷 및 검증 파일 생성 시작...")

    # 0. RESET Packet
    pkt_reset = gen.make_packet(gen.CMD_RESET, target_id=0)
    gen.save_hex_file("00_RESET_OP.txt", pkt_reset)

    # 1. ECHO Packet (Random Data)
    echo_payload = [random.randint(0, 0xFFFF) for _ in range(8)]
    pkt_echo = gen.make_packet(gen.CMD_ECHO, target_id=0, data_list=echo_payload)
    gen.save_hex_file("01_ECHO.txt", pkt_echo)
    print("------------------------------------------------")

    # 2. Write Mem 1: Row Ptr
    pkt_w1 = gen.make_packet(gen.CMD_WRITE, target_id=1, data_list=row_ptr)
    gen.save_hex_file("02_WRITE_MEM1.txt", pkt_w1)

    # 3. Write Mem 2: CSR Data (Normal) & Dense Weights (Optional)
    pkt_w2 = gen.make_packet(gen.CMD_WRITE, target_id=2, data_list=csr_data)
    gen.save_hex_file("03_WRITE_MEM2.txt", pkt_w2)

    # (Optional: Dense Weight Write Test)
    pkt_w2d = gen.make_packet(gen.CMD_WRITE, target_id=2, data_list=packed_sparse_a)
    gen.save_hex_file("03_WRITE_MEM2_d.txt", pkt_w2d)

    # 4. Write Mem 3: Input Vector
    pkt_w3 = gen.make_packet(gen.CMD_WRITE, target_id=3, data_list=packed_dense)
    gen.save_hex_file("04_WRITE_MEM3.txt", pkt_w3)
    print("------------------------------------------------")

    # 5. Read Mem 1: Row Ptr (Request & Expect)
    pkt_r1 = gen.make_packet(gen.CMD_READ, target_id=1, length_val=len(row_ptr))
    gen.save_hex_file("05_READ_MEM1.txt", pkt_r1)
    save_expect_file("05_READ_MEM1_EXPECT.txt", row_ptr)

    # 6. Read Mem 2: CSR Data (Request & Expect)
    pkt_r2 = gen.make_packet(gen.CMD_READ, target_id=2, length_val=len(csr_data))
    gen.save_hex_file("06_READ_MEM2.txt", pkt_r2)
    save_expect_file("06_READ_MEM2_EXPECT.txt", csr_data)

    # (Optional: Dense Weight Read Check)
    pkt_r2d = gen.make_packet(gen.CMD_READ, target_id=2, length_val=len(packed_sparse_a))
    gen.save_hex_file("06_READ_MEM2_d.txt", pkt_r2d)
    save_expect_file("06_READ_MEM2_d_EXPECT.txt", packed_sparse_a)

    # 7. Read Mem 3: Input Vector
    pkt_r3 = gen.make_packet(gen.CMD_READ, target_id=3, length_val=len(packed_dense))
    gen.save_hex_file("07_READ_MEM3.txt", pkt_r3)
    save_expect_file("07_READ_MEM3_EXPECT.txt", packed_dense)
    print("------------------------------------------------")

    # 8. Start Operation
    pkt_start = gen.make_packet(gen.CMD_START, target_id=0)
    gen.save_hex_file("08_START_OP.txt", pkt_start)

    # 9. Read Result C
    # 결과 길이 = Rows (32)
    pkt_res = gen.make_packet(gen.CMD_READ, target_id=4, length_val=len(ref_c))
    gen.save_hex_file("09_READ_RES.txt", pkt_res)

    # Golden Result Expect (32-bit!)
    save_expect_file("09_READ_RES_EXPECT.txt", ref_c, is_32bit=True)

    # 10. Read Perf
    pkt_perf = gen.make_packet(gen.CMD_PERF, target_id=0, length_val=1)
    gen.save_hex_file("10_READ_PERF.txt", pkt_perf)

    print("\n✨ 모든 패킷 및 검증 파일 생성 완료!")
    print(f"📂 저장 위치: {os.path.abspath(PACKET_DIR)}")

    # ====================================================
    # [Step 4] 사람이 보기 편한 메모리 뷰 변환
    # ====================================================
    print("\n[Step 4] Human-Readable View 변환 중...")

    # Row Ptr (16-bit)
    convert_byte_to_word_format("05_READ_MEM1_EXPECT.txt", 2)
    # CSR Data (16-bit)
    convert_byte_to_word_format("06_READ_MEM2_EXPECT.txt", 2)
    # Input Vector (16-bit)
    convert_byte_to_word_format("07_READ_MEM3_EXPECT.txt", 2)
    # Result C (32-bit) -> 중요!
    convert_byte_to_word_format("09_READ_RES_EXPECT.txt", 4)
    # (Optional) Dense Weights
    convert_byte_to_word_format("06_READ_MEM2_d_EXPECT.txt", 2)

# --- 실행 ---
# (앞선 코드들의 model, testloader가 정의되어 있어야 함)
generate_full_test_scenario(model, testloader, frac_bits=4)

In [ ]:
import torch
import numpy as np
import os
import random

# 저장 경로 설정
PACKET_DIR = "uart_packets_generated"
if not os.path.exists(PACKET_DIR):
    os.makedirs(PACKET_DIR)

# ====================================================
# 1. Helper Classes & Functions
# ====================================================

class UARTPacketGenerator:
    """ UART 패킷 생성기 (사용자 정의 프로토콜 준수) """
    def __init__(self, header=0xAA):
        self.header = header
        self.CMD_WRITE = 0x1
        self.CMD_READ  = 0x2
        self.CMD_RESET = 0x3
        self.CMD_START = 0x4
        self.CMD_ECHO  = 0x5
        self.CMD_PERF  = 0x6

    def _pack_cmd_id(self, cmd, target_id):
        return ((cmd & 0x0F) << 4) | (target_id & 0x0F)

    def make_packet(self, cmd, target_id, length_val=0, data_list=None):
        packet = []
        packet.append(self.header)
        packet.append(self._pack_cmd_id(cmd, target_id))

        actual_len = len(data_list) if data_list else length_val
        packet.append(actual_len & 0xFF)
        packet.append((actual_len >> 8) & 0xFF)

        if data_list:
            for data in data_list:
                packet.append(data & 0xFF)
                packet.append((data >> 8) & 0xFF)
        return packet

    def save_hex_file(self, filename, byte_stream):
        path = os.path.join(PACKET_DIR, filename)
        with open(path, 'w') as f:
            for byte in byte_stream:
                f.write(f"{byte:02X}\n")
        print(f"💾 {filename:<25} 저장 완료 ({len(byte_stream)} bytes)")

class SpMMDataProcessor:
    """ 데이터 변환 및 패킹 클래스 """
    def __init__(self, rows=32, cols=50):
        self.rows = rows
        self.cols = cols

    def to_csr(self, sparse_matrix):
        row_ptr = [0]
        csr_data = []
        for row in range(self.rows):
            nnz_in_row = 0
            for col in range(self.cols):
                val = sparse_matrix[row, col]
                if val != 0:
                    packed_val = ((int(val) & 0xFF) << 8) | (col & 0xFF)
                    csr_data.append(packed_val)
                    nnz_in_row += 1
            row_ptr.append(row_ptr[-1] + nnz_in_row)
        return row_ptr, csr_data

    def pack_dense_vector(self, dense_vector):
        """ 1D 벡터 패킹 (2 elements -> 1 word) """
        packed = []
        vec = dense_vector.tolist() if isinstance(dense_vector, np.ndarray) else dense_vector
        if len(vec) % 2 != 0: vec.append(0) # Padding

        for j in range(0, len(vec), 2):
            low = int(vec[j]) & 0xFF
            high = int(vec[j+1]) & 0xFF
            packed.append((high << 8) | low)
        return packed

    def pack_dense_matrix(self, matrix):
        """ 2D 매트릭스 패킹 (Dense Weight Test용) """
        packed = []
        # Row-Major 순회하며 2개씩 묶음
        flat = matrix.flatten()
        return self.pack_dense_vector(flat)

def convert_byte_to_word_format(input_file_, bytes_per_word=2):
    """ 바이트 파일을 읽어 사람이 보기 편한 메모리 뷰 생성 """
    input_path = os.path.join(PACKET_DIR, input_file_)
    if not os.path.exists(input_path):
        print(f"❌ 파일 없음: {input_path}")
        return

    output_file = input_path.replace(".txt", "_MEM_VIEW.txt")

    try:
        with open(input_path, 'r') as f_in, open(output_file, 'w') as f_out:
            lines = [line.strip() for line in f_in.readlines() if line.strip()]

            f_out.write(f"# Memory View for {input_file_}\n")
            f_out.write(f"# Width: {bytes_per_word * 8}-bit\n")
            f_out.write(f"# {'Address':<8} | {'Hex Value':<10} | {'Decimal'}\n")
            f_out.write("-" * 45 + "\n")

            word_count = 0
            for i in range(0, len(lines), bytes_per_word):
                chunks = lines[i : i + bytes_per_word]
                if len(chunks) < bytes_per_word: break

                # Little Endian Logic
                hex_str = ""
                for b in reversed(chunks): hex_str += b

                # Signed Int Check (32bit vs 16bit)
                val_int = int(hex_str, 16)
                if bytes_per_word == 4 and val_int > 0x7FFFFFFF:
                    val_int -= 0x100000000
                elif bytes_per_word == 2 and val_int > 0x7FFF:
                    val_int -= 0x10000

                f_out.write(f"@{word_count:04X}     | {hex_str}       | {val_int}\n")
                word_count += 1

        print(f"✅ 뷰 생성 완료: {os.path.basename(output_file)}")
    except Exception as e:
        print(f"❌ 뷰 생성 실패: {e}")

# ====================================================
# 2. Main Logic: PyTorch Data -> Packets
# ====================================================

def generate_full_test_scenario(model, testloader, frac_bits=4):
    gen = UARTPacketGenerator()
    device = next(model.parameters()).device
    model.eval()

    # --- [Step 1] PyTorch 데이터 추출 ---
    print("\n[Step 1] PyTorch 모델에서 데이터 추출 및 양자화...")
    images, _ = next(iter(testloader))
    x = images[0].view(-1, 28 * 28).to(device)
    scale = 2 ** frac_bits

    with torch.no_grad():
        # Layer 1 Sim
        x_int = torch.clamp(torch.round(x * scale), -128, 127)
        x_sim = x_int.float() / scale
        w1_int = torch.clamp(torch.round(model.fc1.weight * scale), -128, 127)
        w1_sim = w1_int.float() / scale
        out1 = torch.nn.functional.relu(torch.nn.functional.linear(x_sim, w1_sim, model.fc1.bias))

        # Layer 2 Input (Vector B)
        input_vec_int = torch.clamp(torch.round(out1.view(-1) * scale), -128, 127).long().cpu().numpy()

        # Layer 2 Weight (Matrix A)
        weight_mat_int = torch.clamp(torch.round(model.fc2.weight * scale), -128, 127).long().cpu().numpy()

        # Golden Result (Ref C)
        # (1, 50) x (50, 32) + Bias -> 32 elements
        # Integer MAC Operation
        golden_res_int = np.dot(input_vec_int, weight_mat_int.T)
        bias_int = torch.round(model.fc2.bias * (scale**2)).detach().cpu().numpy()
        ref_c = (golden_res_int + bias_int).astype(np.int32)

    # --- [Step 2] 데이터 패킹 (CSR, Dense) ---
    print("[Step 2] 데이터를 하드웨어 포맷으로 변환...")
    rows, cols = weight_mat_int.shape
    proc = SpMMDataProcessor(rows, cols)

    # 1. CSR Data
    row_ptr, csr_data = proc.to_csr(weight_mat_int)

    # 2. Dense Packed Weight (Optional: 'd' suffix)
    packed_sparse_a = proc.pack_dense_matrix(weight_mat_int)

    # 3. Packed Input Vector
    packed_dense = proc.pack_dense_vector(input_vec_int)

    # --- Helper: Expect 파일 저장 ---
    def save_expect_file(filename, data_list, is_32bit=False):
        byte_stream = []
        for val in data_list:
            val = int(val)
            byte_stream.append(val & 0xFF)
            byte_stream.append((val >> 8) & 0xFF)
            if is_32bit:
                byte_stream.append((val >> 16) & 0xFF)
                byte_stream.append((val >> 24) & 0xFF)
        gen.save_hex_file(filename, byte_stream)
        # .mem 파일도 동일하게 생성
        gen.save_hex_file(filename.replace(".txt", ".mem"), byte_stream)


    # ====================================================
    # [Step 3] UART 패킷 생성 시나리오
    # ====================================================
    print("\n[Step 3] UART 패킷 및 검증 파일 생성 시작...")

    # 0. RESET Packet
    pkt_reset = gen.make_packet(gen.CMD_RESET, target_id=0)
    gen.save_hex_file("00_RESET_OP.txt", pkt_reset)

    # 1. ECHO Packet (Random Data)
    echo_payload = [random.randint(0, 0xFFFF) for _ in range(8)]
    pkt_echo = gen.make_packet(gen.CMD_ECHO, target_id=0, data_list=echo_payload)
    gen.save_hex_file("01_ECHO.txt", pkt_echo)
    print("------------------------------------------------")

    # 2. Write Mem 1: Row Ptr
    pkt_w1 = gen.make_packet(gen.CMD_WRITE, target_id=1, data_list=row_ptr)
    gen.save_hex_file("02_WRITE_MEM1.txt", pkt_w1)

    # 3. Write Mem 2: CSR Data (Normal) & Dense Weights (Optional)
    pkt_w2 = gen.make_packet(gen.CMD_WRITE, target_id=2, data_list=csr_data)
    gen.save_hex_file("03_WRITE_MEM2.txt", pkt_w2)

    # (Optional: Dense Weight Write Test)
    pkt_w2d = gen.make_packet(gen.CMD_WRITE, target_id=2, data_list=packed_sparse_a)
    gen.save_hex_file("03_WRITE_MEM2_d.txt", pkt_w2d)

    # 4. Write Mem 3: Input Vector
    pkt_w3 = gen.make_packet(gen.CMD_WRITE, target_id=3, data_list=packed_dense)
    gen.save_hex_file("04_WRITE_MEM3.txt", pkt_w3)
    print("------------------------------------------------")

    # 5. Read Mem 1: Row Ptr (Request & Expect)
    pkt_r1 = gen.make_packet(gen.CMD_READ, target_id=1, length_val=len(row_ptr))
    gen.save_hex_file("05_READ_MEM1.txt", pkt_r1)
    save_expect_file("05_READ_MEM1_EXPECT.txt", row_ptr)

    # 6. Read Mem 2: CSR Data (Request & Expect)
    pkt_r2 = gen.make_packet(gen.CMD_READ, target_id=2, length_val=len(csr_data))
    gen.save_hex_file("06_READ_MEM2.txt", pkt_r2)
    save_expect_file("06_READ_MEM2_EXPECT.txt", csr_data)

    # (Optional: Dense Weight Read Check)
    pkt_r2d = gen.make_packet(gen.CMD_READ, target_id=2, length_val=len(packed_sparse_a))
    gen.save_hex_file("06_READ_MEM2_d.txt", pkt_r2d)
    save_expect_file("06_READ_MEM2_d_EXPECT.txt", packed_sparse_a)

    # 7. Read Mem 3: Input Vector
    pkt_r3 = gen.make_packet(gen.CMD_READ, target_id=3, length_val=len(packed_dense))
    gen.save_hex_file("07_READ_MEM3.txt", pkt_r3)
    save_expect_file("07_READ_MEM3_EXPECT.txt", packed_dense)
    print("------------------------------------------------")

    # 8. Start Operation
    pkt_start = gen.make_packet(gen.CMD_START, target_id=0)
    gen.save_hex_file("08_START_OP.txt", pkt_start)

    # 9. Read Result C
    # 결과 길이 = Rows (32)
    pkt_res = gen.make_packet(gen.CMD_READ, target_id=4, length_val=len(ref_c))
    gen.save_hex_file("09_READ_RES.txt", pkt_res)

    # Golden Result Expect (32-bit!)
    save_expect_file("09_READ_RES_EXPECT.txt", ref_c, is_32bit=True)

    # 10. Read Perf
    pkt_perf = gen.make_packet(gen.CMD_PERF, target_id=0, length_val=1)
    gen.save_hex_file("10_READ_PERF.txt", pkt_perf)

    print("\n✨ 모든 패킷 및 검증 파일 생성 완료!")
    print(f"📂 저장 위치: {os.path.abspath(PACKET_DIR)}")

    # ====================================================
    # [Step 4] 사람이 보기 편한 메모리 뷰 변환
    # ====================================================
    print("\n[Step 4] Human-Readable View 변환 중...")

    # Row Ptr (16-bit)
    convert_byte_to_word_format("05_READ_MEM1_EXPECT.txt", 2)
    # CSR Data (16-bit)
    convert_byte_to_word_format("06_READ_MEM2_EXPECT.txt", 2)
    # Input Vector (16-bit)
    convert_byte_to_word_format("07_READ_MEM3_EXPECT.txt", 2)
    # Result C (32-bit) -> 중요!
    convert_byte_to_word_format("09_READ_RES_EXPECT.txt", 4)
    # (Optional) Dense Weights
    convert_byte_to_word_format("06_READ_MEM2_d_EXPECT.txt", 2)

# --- 실행 ---
# (앞선 코드들의 model, testloader가 정의되어 있어야 함)
generate_full_test_scenario(model, testloader, frac_bits=4)